In [1]:
import os
import json
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
# from key import OPENAI_KEY, LANGSMITH_KEY       # Import your own keys
from key import DASHSCOPE_API_KEY, LANGSMITH_KEY      # Add your own keys
from copy import deepcopy
import time
from tqdm import tqdm

# os.environ["OPENAI_API_KEY"] = OPENAI_KEY
os.environ["DASHSCOPE_API_KEY"] = DASHSCOPE_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_KEY

In [2]:
# text_dir = "../data/text/Subtask_2_test.json"
# anno = json.load(open(text_dir))

text_dir = "../data/text/Subtask_2_train.json"
anno = json.load(open(text_dir))
print("anno[:78] :", anno[:78])
anno = anno[:78]
video_captions = json.load(open("eval_proc_out.json"))
cause_windows = json.load(open("cause_windows.json"))
# emotion_labels = json.load(open("emotion_eval_labelled.json"))
emotion_labels = json.load(open("emotion_eval_labelled1.json"))

anno[:78] : [{'conversation_ID': 1, 'conversation': [{'utterance_ID': 1, 'text': 'Alright , so I am back in high school , I am standing in the middle of the cafeteria , and I realize I am totally naked .', 'speaker': 'Chandler', 'emotion': 'neutral', 'video_name': 'dia1utt1.mp4'}, {'utterance_ID': 2, 'text': 'Oh , yeah . Had that dream .', 'speaker': 'All', 'emotion': 'neutral', 'video_name': 'dia1utt2.mp4'}, {'utterance_ID': 3, 'text': 'Then I look down , and I realize there is a phone ... there .', 'speaker': 'Chandler', 'emotion': 'surprise', 'video_name': 'dia1utt3.mp4'}, {'utterance_ID': 4, 'text': 'Instead of ... ?', 'speaker': 'Joey', 'emotion': 'surprise', 'video_name': 'dia1utt4.mp4'}, {'utterance_ID': 5, 'text': 'That is right .', 'speaker': 'Chandler', 'emotion': 'anger', 'video_name': 'dia1utt5.mp4'}, {'utterance_ID': 6, 'text': 'Never had that dream .', 'speaker': 'Joey', 'emotion': 'neutral', 'video_name': 'dia1utt6.mp4'}, {'utterance_ID': 7, 'text': 'No .', 'speaker': 'P

In [3]:
### 查看openai版本号
import openai
print(openai.__version__)

1.109.1


#### Postprocessing Emotion Labelled data to ensure valid emotions

In [4]:
emotions = ["anger", "joy", "sadness", "surprise", "disgust", "fear"]

for a in anno:
    a["emotion-cause_pairs"] = []
    emo_labels = emotion_labels[str(a["conversation_ID"])]["conversation"]
    for i, utt in enumerate(a["conversation"]):
        emo = "neutral"
        try:
            emo = emo_labels[i]["emotion"].lower()
            if emo not in emotions: emo = "neutral"
        except:
            emo = "neutral"
        utt["emotion"] = emo

#### Loading Cause index and Window Functions

In [5]:
from langchain_community.embeddings import DashScopeEmbeddings

embeddings = DashScopeEmbeddings(
                    # model="text-embedding-v3",  # 或者你实际使用的模型
                    model="text-embedding-v4",  # 或者你实际使用的模型
                    dashscope_api_key=DASHSCOPE_API_KEY)

pos = ["beg", "mid", "end"]
db_dict = {emo : {} for emo in emotions}
for emo in emotions:
    for p in pos:
        db_dict[emo][p] = FAISS.load_local(f"cause_windows/{emo}/{p}", embeddings)

In [6]:
def get_window_beg(convo: list, size:int = 3) -> list:
    return deepcopy(convo[:size])

def get_window_end(convo: list, size:int = 6) -> list:
    return deepcopy(convo[-size:])

def get_window_mid(convo:list, idx:int, prev_size:int = 5, next_size:int = 2) -> list:
    return deepcopy(convo[max(0, idx-prev_size) : (idx+1) + next_size])

def format_window(window: tuple, label:bool = False) -> str:
    idx, window = window
    utt_idx = idx
    emo = None
    speaker = None
    out_str = ""

    for i, utt in enumerate(window):
        if idx == utt["utterance_ID"]:
            emo = utt["emotion"]
            speaker = utt["speaker"]
            out_str += f'{i+1}. {utt["speaker"]}: {utt["text"]}'
            if label: out_str += f' [{emo}]\n'
            else: out_str += '\n'
            utt_idx = i+1
        else:
            out_str += f'{i+1}. {utt["speaker"]}: {utt["text"]}\n'
    
    if label:
        out_str += f"\nWhat are the causal utterances that trigger the emotion of {emo} in {speaker} in utterance {utt_idx}?"
    
    return out_str, idx-utt_idx

#### Prompts and RAG Pipeline

In [7]:
explaination_text = """
You are an expert in analyzing conversations to extract the causes of emotions 
in particular utterances by speakers. You give definite confident answers only.

Description of emotional causes:
- Each utterance always has a reason of why it was said and why it had a particular emotion.
- A cause is an utterance that comes before or after the particular utterance in question that best explains to be the reason behind the particular emotion.
- The emotional utterance itself can be a cause of itself if its content ALSO best explains the reason for the particular emotion. 
- Sometimes the cause can be beyond the context of the conversation thus an utterance might have no cause within conversation.
- There can be multiple causes for an utterance.

Here's a conversation:
{conversation}

Analyze and justify the above annotation concisely.
"""

cause_text = """
You are an expert in analyzing conversations to extract the causes of emotions 
in particular utterances by speakers. You give definite confident answers only.

Description of emotional causes:
- Each utterance always has a reason of why it was said and why it had a particular emotion.
- A cause is an utterance that comes before or after the particular utterance in question that best explains to be the reason behind the particular emotion.
- The emotional utterance itself can be a cause of itself if its content ALSO best explains the reason for the particular emotion. 
- Sometimes the cause can be beyond the context of the conversation thus an utterance might have no cause within conversation.
- There can be multiple causes for an utterance.

Here are some examples of how to recgonize causes:
Example 1:
{example_1}

Example 2:
{example_2}

Example 3:
{example_3}

Now, please recognize the causes in following conversation. Heres the context for the whole conversation:
{scene}

Conversation:
{window}
"""

json_text = "{prompt}\n\nReformat the text to JSON as {{'causes': [list of causal utterance numbers]}}. No plain text."

In [8]:
model = ChatOpenAI(
                   # model="qwen-plus", 
                   # model="qwen3.7-flash",
                   # model="qwen3.7-plus",
                   # model="qwen3.7-max",
                   # model="qwen3.8-max",
                   model="qwen3.8-2.4t-a95b",
                   # model="qwen3.8-27b",
                   # model="qwen3.8-omni-flash",
                   # model="qwen3.8-omni-flash-realtime",
                   openai_api_key=DASHSCOPE_API_KEY,
                   base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")
output_parser = StrOutputParser()

In [9]:
explaination_prompt = ChatPromptTemplate.from_template(explaination_text)
cause_prompt = ChatPromptTemplate.from_template(cause_text)
json_prompt = ChatPromptTemplate.from_template(json_text)

explaination_pipeline = explaination_prompt | model | output_parser
cause_chain = cause_prompt | model | output_parser
cause_json_chain = (
    {"prompt": cause_chain}
    | json_prompt
    | model
    | output_parser
)

#### Getting Cause for all Conversations

In [10]:
# cause_eval_labelled = {}
# cause_eval_labelled = json.load(open("cause_eval_labelled.json")) # if resuming
cause_eval_labelled = json.load(open("cause_eval_labelled1.json")) # if resuming

In [11]:
# step = 10
step = 1
for k, a in enumerate(anno):
    conv_idx = a["conversation_ID"]
    convo = a["conversation"]
    for i, utt in enumerate(convo):
        emo = utt["emotion"]
        utt_idx = utt["utterance_ID"]
        name = f"dia{conv_idx}utt{utt_idx}"
        if emo == "neutral": continue
        elif emo not in ["anger", "joy", "sadness", "surprise", "disgust", "fear"]:
            print("  [{}/{}] Failed to process utterance {} due to invalid emotion {}".format(i+1, len(convo), name, emo))
            continue
        if name in cause_eval_labelled: continue
        
        if i == 0:
            pos = "beg"
            window = get_window_beg(convo)
        elif i == len(convo) - 1:
            pos = "end"
            window = get_window_end(convo)
        else:
            pos = "mid"
            window = get_window_mid(convo, i)
        
        try:
            window_str, diff = format_window((utt_idx, window), False)
            window_str_labelled, diff = format_window((utt_idx, window), True)
            closest_windows = db_dict[emo][pos].similarity_search(window_str)[1:4]
            closest_indices = [wdw.page_content.split("\n")[0].strip() for wdw in closest_windows]
            closest_windows = [cause_windows[idx] for idx in closest_indices]
            window_batch = [{"conversation": window} for window in closest_windows]
            # explainations = explaination_pipeline.batch(window_batch, config={"max_concurrency": 3})       
            explainations = explaination_pipeline.batch(window_batch, config={"max_concurrency": 1})  
            examples = [closest_windows[i] + "\n" +exp for i, exp in enumerate(explainations)]
            scene = video_captions[str(conv_idx)]
            out = cause_json_chain.invoke({"window": window_str_labelled,
                                        "scene": scene,
                                        "example_1": examples[0],
                                        "example_2": examples[1],
                                        "example_3": examples[2]})
            causes = json.loads(out)["causes"]
            ecp = []
            for c in causes:
                ecp.append([f"{utt_idx}_{emo}", str(int(c)+diff)])
            a["emotion-cause_pairs"].extend(ecp)
            cause_eval_labelled[name] = ecp   
                
            print("  [{}/{}] Processed utterance {}".format(i+1, len(convo), name))
            
        except Exception as error:
             print("  [{}/{}] Failed to process utterance {} due to {}".format(i+1, len(convo), name, error))
             
    print("[{}/{}] Processed Conv {}".format(k+1, len(anno), conv_idx))
    if k % step == 0:
        print("json dump...")
        # json.dump(cause_eval_labelled, open("cause_eval_labelled.json", "w"))
        json.dump(cause_eval_labelled, open("cause_eval_labelled1.json", "w"))

[1/78] Processed Conv 1
json dump...
[2/78] Processed Conv 2
json dump...
[3/78] Processed Conv 3
json dump...
[4/78] Processed Conv 4
json dump...
[5/78] Processed Conv 5
json dump...
[6/78] Processed Conv 6
json dump...
[7/78] Processed Conv 7
json dump...
[8/78] Processed Conv 8
json dump...
[9/78] Processed Conv 9
json dump...
[10/78] Processed Conv 10
json dump...
[11/78] Processed Conv 11
json dump...
[12/78] Processed Conv 12
json dump...
[13/78] Processed Conv 13
json dump...
[14/78] Processed Conv 14
json dump...
[15/78] Processed Conv 15
json dump...
[16/78] Processed Conv 17
json dump...
[17/78] Processed Conv 19
json dump...
[18/78] Processed Conv 21
json dump...
[19/78] Processed Conv 22
json dump...
[20/78] Processed Conv 24
json dump...
[21/78] Processed Conv 25
json dump...
[22/78] Processed Conv 27
json dump...
[23/78] Processed Conv 29
json dump...
[24/78] Processed Conv 30
json dump...
[25/78] Processed Conv 31
json dump...
[26/78] Processed Conv 32
json dump...
[27/

Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=75d5fb32-4425-4fb6-bfc7-50cbfaaddd20,id=75d5fb32-4425-4fb6-bfc7-50cbfaaddd20; trace=75d5fb32-4425-4fb6-bfc7-50cbfaaddd20,id=5cf4fd6b-b780-4a64-9e0b-00bd679ea261; trace=75d5fb32-4425-4fb6-bfc7-50cbfaaddd20,id=251e6084-8ebb-4f9e-be34-e7745fd4b08c; trace=ee951f5f-2260-4a27-8fb3-a598183aeb0d,id=ee951f5f-2260-4a27-8fb3-a598183aeb0d; trace=ee951f5f-2260-4a27-8fb3-a598183aeb0d,id=5a6c0e0a-5a88-4d0b-a5a3-a251d611d646; trace=4bbe60a1-1b33-497e-9192-1f94a417742e,id=4bbe60a1-1b33-497e-9192-1f94a417742e; trace=4bbe60a1-1b33-497e-9192-1f94a417742e,id=f6032566-6b66-4fd3-af2c-435b3e46a35e
Failed to multipart ingest runs: langsmith.utils.LangS

  [2/8] Processed utterance dia102utt2


Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=db509f1c-725a-4e98-9ab2-ba68d21edf54,id=fa3fee92-7b8f-41a2-b4a6-1c7cd507d5ba; trace=db509f1c-725a-4e98-9ab2-ba68d21edf54,id=db509f1c-725a-4e98-9ab2-ba68d21edf54; trace=db509f1c-725a-4e98-9ab2-ba68d21edf54,id=f9602163-cc8b-434e-89f8-b0e8142cd3c5
Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=66ee175b-de36-4f15-9065-8

  [3/8] Processed utterance dia102utt3


Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=2f2a601c-21eb-498a-92ad-433351014591,id=e9be679f-8022-42fb-8d1d-08a5ad969d5c; trace=2f2a601c-21eb-498a-92ad-433351014591,id=2f2a601c-21eb-498a-92ad-433351014591; trace=2f2a601c-21eb-498a-92ad-433351014591,id=dea80504-e177-4206-9203-6f61e7008d6f
Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=30f29a52-1d85-4cda-9c54-4

  [5/8] Processed utterance dia102utt5


Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=73fdf8ab-88a1-4d3f-9c86-2b09084b804a,id=105cd9bf-48ef-4f00-839e-748310290601; trace=73fdf8ab-88a1-4d3f-9c86-2b09084b804a,id=73fdf8ab-88a1-4d3f-9c86-2b09084b804a; trace=73fdf8ab-88a1-4d3f-9c86-2b09084b804a,id=e0a9378e-44a3-4c42-be03-7c8993c9a866
Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=b3cf61bc-1a0a-4b62-bd48-9

  [7/8] Processed utterance dia102utt7
[78/78] Processed Conv 102
json dump...


Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=d2282db5-5723-496b-bce5-3fe3e3a084d4,id=2f82da68-cb71-4311-b000-c65722be3aab; trace=d2282db5-5723-496b-bce5-3fe3e3a084d4,id=d2282db5-5723-496b-bce5-3fe3e3a084d4; trace=d2282db5-5723-496b-bce5-3fe3e3a084d4,id=81a85985-28b2-4e82-9368-6a183835b0fe


In [12]:
# json.dump(cause_eval_labelled, open("cause_eval_labelled.json", "w"))
json.dump(cause_eval_labelled, open("cause_eval_labelled1.json", "w"))
# json.dump(anno, open("cur_anno.json", "w"))
json.dump(anno, open("cur_anno1.json", "w"))

#### Postprocessing Causes to added self-causes

In [13]:
for convo in anno:
    same_ecp = []
    for utt in convo["conversation"]:
        uid = utt["utterance_ID"]
        emo = utt["emotion"]
        if emo != "neutral":
            same_ecp.append([str(uid)+"_"+emo, str(uid)])
    for p in same_ecp:
        if p not in convo["emotion-cause_pairs"]:
            convo["emotion-cause_pairs"].append(p)

# json.dump(anno, open("cur_anno_same_added.json", "w"))
json.dump(anno, open("cur_anno_same_added1.json", "w"))